In [ ]:
# generated by axis/splitcausal_t4probe.py -- do not edit by hand
import base64, hashlib, pathlib
_PAYLOAD = '\n# ---------------------------------------------------------------- tinygrad FIRST\n#\n# THE ORDER OF THESE LINES IS THE WHOLE REASON THIS SECTION EXISTS. The first\n# version of this notebook imported `tinygrad` inside S0, and the kernel died in\n# 9.5 s with:\n#\n#     ModuleNotFoundError: No module named \'tinygrad\'\n#\n# The Kaggle image ships `papermill` and `nbconvert` but NOT tinygrad.\n# `axis/kaggle/make_notebook.py` installs it in its own cell, and this probe has\n# no such cell, so the import has to be done here. It is at the very top, before\n# anything else, so a missing tinygrad fails in 30 seconds having printed a\n# sentence saying what is missing, rather than at the end of a run.\nimport subprocess, sys\nif "tinygrad" not in sys.modules:\n    try:\n        import tinygrad  # noqa: F401\n    except ImportError:\n        print("tinygrad is not in this image; installing it", flush=True)\n        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tinygrad"])\n\nimport hashlib, json, os, re, statistics, time\nimport numpy as np\n\nDEV = os.environ.get("DEV", "CUDA")\nBUDGET_S = float(os.environ.get("PROBE_BUDGET_S", "1000"))\nT0 = time.time()\n\ndef log(*a):\n    print(*a, flush=True)\n\ndef left():\n    return BUDGET_S - (time.time() - T0)\n\ndef relerr(a, want):\n    a = np.asarray(a, np.float64)\n    return float(np.abs(a - want).max() / max(1e-30, np.abs(want).max()))\n\n# ---------------------------------------------------------------- S0 provenance\nlog("=" * 78)\nlog("S0  PROVENANCE")\nlog("=" * 78)\nimport tinygrad\nfrom tinygrad import Device, Tensor, GlobalCounters\nimport tinygrad.mixin.op as M\nimport tinygrad.tensor as TWMOD\n\nOP_PY = __import__("pathlib").Path(M.__file__)\nSRC = OP_PY.read_text()\n_h = hashlib.sha256(OP_PY.read_bytes()).hexdigest()\n_m = re.search(r"^(\\s*)SPLIT = (\\d+)\\s*$", SRC, re.M)\nassert _m, "SPLIT literal not found in the installed file"\nINSTALLED = int(_m.group(2))\nORIG_SRC = __import__("inspect").getsource(M.OpMixin._split_cumalu)\nSPLIT_RE = re.compile(r"^(\\s*)SPLIT = (\\d+)\\s*$", re.M)\n\nlog("  tinygrad      ", getattr(tinygrad, "__version__", "?"), "|", tinygrad.__file__)\nlog("  op.py         ", OP_PY)\nlog("  op.py sha256  ", _h)\nlog("  installed SPLIT", INSTALLED, "-> source predicts the first two-stage N at", 2*INSTALLED+1)\nlog("  Tensor wraps its mixin methods:",\n    Tensor.__dict__.get("_split_cumalu") is not M.OpMixin.__dict__["_split_cumalu"])\nfor a in ("name", "capability", "driver_version", "arch", "clock_hz"):\n    try: log(f"  {a:<14}", getattr(Device.DEFAULT, a))\n    except Exception: pass\ntry:\n    from tinygrad.runtime import ops_cuda\n    log("  device prop   ", ops_cuda.cuda_props())\nexcept Exception as e:\n    log("  device prop    unavailable:", type(e).__name__, e)\ntry:\n    import subprocess as _sp\n    log("  nvcc           ", _sp.run(["nvcc", "--version"], capture_output=True,\n                                    text=True).stdout.strip().splitlines()[-1])\nexcept Exception as e:\n    log("  nvcc            unavailable:", type(e).__name__, e)\n\n# ------------------------------------------------------------- instrument\ndef require_fresh(t, what):\n    u = getattr(t, "uop", None)\n    if u is None: return\n    try:\n        fresh = (not u.is_virtual) and (not u.has_buffer_identity())\n    except Exception:\n        return\n    if not fresh:\n        raise RuntimeError(f"{what}: tensor is ALREADY REALIZED; realizing it again "\n                           f"is a silent no-op and timing it measures nothing. "\n                           f"Rebuild the graph (or use time_call on the LINEAR).")\n\ndef linear_device_time_ms(tensor, iters=3):\n    """Re-derive of minecu_tg/bench.py:linear_device_time_ms.\n\n    Walks the compiled LINEAR and re-executes each CALL with tinygrad\'s own\n    time_call, then SUMS the per-kernel device times. Immune to the\n    already-realized no-op, because it never calls realize().\n    """\n    from tinygrad.engine.realize import time_call\n    lin, vv = Tensor.linear_with_vars(tensor)\n    calls = list(lin.src)\n    if not calls: return None\n    for _ in range(2):\n        for c in calls: next(time_call(c, var_vals=vv))\n    tot = []\n    for _ in range(iters):\n        s = 0.0\n        for c in calls: s += next(time_call(c, var_vals=vv))\n        tot.append(s * 1e3)\n    tot.sort()\n    return {"n_kernels": len(calls), "median_ms": statistics.median(tot),\n            "min_ms": tot[0], "max_ms": tot[-1],\n            "per_kernel_us": [round(next(time_call(c, var_vals=vv))*1e6, 2) for c in calls]}\n\n# ------------------------------------------------------------- the patch\nimport functools, textwrap, difflib\n_ORIG_TENSOR = Tensor._split_cumalu\n\ndef patch_split(k):\n    import importlib\n    tinygrad.codegen.to_program_cache.clear()\n    new_src, n = SPLIT_RE.subn(lambda m: f"{m.group(1)}SPLIT = {k}", ORIG_SRC, count=1)\n    if n != 1 or f"SPLIT = {k}" not in new_src:\n        raise RuntimeError(f"the SPLIT substitution did not take exactly once (n={n})")\n    body = textwrap.dedent(new_src)\n    ns = dict(vars(M))\n    exec(compile(body, "<probe:patched _split_cumalu>", "exec"), ns)\n    fn = ns["_split_cumalu"]\n    Tensor._split_cumalu = functools.wraps(fn)(TWMOD._metadata_wrapper(fn))\n    d = list(difflib.unified_diff(ORIG_SRC.splitlines(), new_src.splitlines(),\n                                  fromfile=f"installed SPLIT={INSTALLED}",\n                                  tofile=f"patched SPLIT={k}", lineterm="", n=1))\n    return {"split": k, "substitutions": n, "diff": d}\n\ndef restore_split():\n    Tensor._split_cumalu = _ORIG_TENSOR\n    import tinygrad.codegen\n    tinygrad.codegen.to_program_cache.clear()\n\ndef clear_caches():\n    import tinygrad.codegen as CG\n    from tinygrad.engine import realize as RZ\n    CG.to_program_cache.clear()\n    for nm in ("runtime_cache", "graph_cache", "local_size_cache"):\n        if hasattr(RZ, nm): getattr(RZ, nm).clear()\n\n# ------------------------------------------------------------- oracle\ndef make_input(r, n, seed=12):\n    rs = np.random.RandomState(seed)\n    return rs.rand(r, n).astype(np.float32)\n\ndef arm_cumsum(x_np):\n    t = Tensor(x_np).cumsum(-1)\n    require_fresh(t, "cumsum arm")\n    return t\n\ndef arm_one_stage(x_np):\n    import tinygrad.mixin.op as _M\n    t = Tensor(x_np)._cumalu(-1, _M.Ops.ADD)\n    require_fresh(t, "one-stage arm")\n    return t\n\ndef time_arm(build, x_np, want, label, iters=3):\n    """Build fresh, GATE against the float64 oracle, then time by re-execution."""\n    g = build(x_np)\n    d = linear_device_time_ms(g, iters)\n    e = relerr(g.numpy(), want)\n    if e >= 1e-3:\n        raise RuntimeError(f"{label}: GATE FAILED, relerr={e:.3e} at {x_np.shape}. "\n                           f"Refusing to time an arm that does not compute cumsum.")\n    return {"label": label, "relerr": e, **d}\n\n# ============================================================ S1  one vs two\nlog("")\nlog("=" * 78)\nlog("S1  THE CHEAP CAUSAL HANDLE: one-stage vs two-stage at ONE fixed shape")\nlog("    no constant moved, no file edited")\nlog("=" * 78)\nN1 = 4096\nE = 1 << 24\nR1 = E // N1\nassert R1 * N1 == E, f"element count is not 2^24: R={R1} N={N1}"\nlog(f"  shape R={R1} N={N1}  E={R1*N1}  (the ladder\'s own shape and element count)")\nx1 = make_input(R1, N1)\nwant1 = np.cumsum(x1.astype(np.float64), -1)\ns1 = {}\ntry:\n    t_cum = time_arm(arm_cumsum, x1, want1, f"cumsum (two-stage, N={N1}>{2*INSTALLED})")\n    t_one = time_arm(arm_one_stage, x1, want1, f"_cumalu (one-stage, N={N1})")\n    s1 = {"R": R1, "N": N1, "elements": R1*N1, "cumsum": t_cum, "one_stage": t_one}\n    log(f"  cumsum  (2-stage) {t_cum[\'median_ms\']:8.3f} ms  {t_cum[\'n_kernels\']} kernels"\n        f"  per-kernel {t_cum[\'per_kernel_us\']}  relerr {t_cum[\'relerr\']:.2e}")\n    log(f"  _cumalu (1-stage) {t_one[\'median_ms\']:8.3f} ms  {t_one[\'n_kernels\']} kernels"\n        f"  per-kernel {t_one[\'per_kernel_us\']}  relerr {t_one[\'relerr\']:.2e}")\n    s1["ratio_two_over_one"] = t_cum["median_ms"] / t_one["median_ms"]\n    log(f"  RATIO two-stage / one-stage = {s1[\'ratio_two_over_one\']:.3f}x")\n    log("  F4: one-stage not faster => the DECOMPOSITION is not the cost."\n        if s1["ratio_two_over_one"] < 1.0 else\n        "  F4 FIRED: two-stage is the slower arm, decomposition could be the cost.")\nexcept Exception as e:\n    s1 = {"error": f"{type(e).__name__}: {e}"}\n    log("  S1 FAILED:", s1["error"])\nlog(f"  [{left():.0f}s of budget left]")\n\n# ================================================== S2 / S3  the density sweep\nDENSE = [256, 320, 384, 448, 480, 512, 513, 544, 640, 768, 896, 1024,\n         1536, 2048, 4096]\n\n# WHY THIS GRID, HONESTLY. It is DENSE where the mechanism is -- every point\n# from 256 to 1024, which is the interval `notes/33` had no point in at all, and\n# the interval the framework\'s own switch at N=512 sits in the middle of. It is\n# SPARSE below 256 and at the far end, because those points are not where the\n# question is and each one costs device time.\n#\n# It is sparse below 256 for a reason worth stating: the reference\n# `axis/refkernels.cu` cannot be launched there at all except through the\n# scalar kernel, so a RATIO against the reference is not available below 256 and\n# the mechanism claim is a claim about the framework\'s own cost. The reference\n# question is a different experiment and is not this one.\n\ndef sweep(split_value, label, iters=3, floor_s=90):\n    if split_value is not None:\n        pa = patch_split(split_value)\n        log(f"  --- patch: {split_value} substitutions, installed on Tensor")\n        for l in pa["diff"]: log("      " + l)\n    else:\n        restore_split()\n    out = []\n    try:\n        log(f"  {\'N\':>6} {\'R\':>7} {\'path\':>10} {\'kernels\':>8} {\'dev ms\':>9} {\'xE*SPLIT\':>9} {\'relerr\':>9}")\n        for n in DENSE:\n            if left() < floor_s:\n                log(f"  N={n} SKIPPED, out of budget ({left():.0f}s left)")\n                continue\n            r = max(1, E // n)\n            clear_caches()\n            x = make_input(r, n)\n            want = np.cumsum(x.astype(np.float64), -1)\n            t0 = time.time()\n            try:\n                t = time_arm(arm_cumsum, x, want, f"cumsum N={n}", iters)\n            except Exception as e:\n                log(f"  {n:>6} {r:>7}  FAILED {type(e).__name__}: {str(e)[:90]}")\n                out.append({"N": n, "R": r, "error": f"{type(e).__name__}: {str(e)[:200]}"})\n                continue\n            sp = split_value if split_value is not None else INSTALLED\n            path = "one-stage" if n <= 2 * sp else "two-stage"\n            unit = E * sp / 1e6\n            out.append({"N": n, "R": r, "path": path, "kernels": t["n_kernels"],\n                        "median_ms": t["median_ms"], "min_ms": t["min_ms"],\n                        "x_predicted": round(t["median_ms"] / unit, 4),\n                        "relerr": t["relerr"], "per_kernel_us": t["per_kernel_us"],\n                        "wall_s": round(time.time() - t0, 1)})\n            log(f"  {n:>6} {r:>7} {path:>10} {t[\'n_kernels\']:>8} "\n                f"{t[\'median_ms\']:>9.3f} {t[\'median_ms\']/unit:>9.3f} {t[\'relerr\']:>9.1e}"\n                f"   [{time.time()-t0:.1f}s]")\n    finally:\n        if split_value is not None:\n            restore_split()\n    return {"label": label, "split": split_value if split_value is not None else INSTALLED,\n            "patch_diff": (pa["diff"] if split_value is not None else None),\n            "points": out}\n\nlog("")\nlog("=" * 78)\nlog("S2  BASELINE density sweep, SPLIT UNMODIFIED (256)")\nlog("    E = 2^24 held constant; notes/33\'s grid had NO point between 256 and 1024")\nlog("=" * 78)\ns2 = sweep(None, "baseline")\nlog(f"  [{left():.0f}s of budget left]")\n\nlog("")\nlog("=" * 78)\nlog("S3  THE SPLIT SWEEP. The plateau must scale with SPLIT; the peak must sit at 2*SPLIT.")\nlog("=" * 78)\nsweeps = {"baseline": s2}\n# THE SPLIT SWEEP IS THE EXPERIMENT, so it gets the budget and the baseline\n# sweep is what yields. The baseline is also re-measured as part of the verdict\n# if time allows, because an unmodified control in the SAME session is what makes\n# a difference a fact rather than a session difference.\nSPLIT_VALUES = [64, 1024]\nfor k in SPLIT_VALUES:\n    if left() < 200:\n        log(f"  SPLIT={k} SKIPPED, out of budget ({left():.0f}s left)")\n        continue\n    log("")\n    log(f"  ---- SPLIT = {k} ----")\n    sweeps[f"split{k}"] = sweep(k, f"split{k}")\n    log(f"  [{left():.0f}s of budget left]")\n\n# The control that makes F1 a fact rather than a session difference: the\n# UNMODIFIED build re-measured AFTER the patched runs, in the same process, on the\n# same card. If the modified N=4096 numbers differ from this one for any reason\n# other than SPLIT, F1 is void and the log says so.\nif left() > 120:\n    log("")\n    log("  ---- control: the UNMODIFIED build, re-measured last, same process ----")\n    ct = sweep(None, "control-last", iters=3, floor_s=60)\n    ct["points"] = [p for p in ct["points"] if p["N"] in (512, 513, 1024, 4096)]\n    sweeps["control_last"] = ct\n    for p in ct["points"]:\n        log(f"    control N={p[\'N\']:<6} {p[\'median_ms\']:8.3f} ms  {p[\'kernels\']} kernels")\n\n# ----------------------------------------------------------------- verdict\nlog("")\nlog("=" * 78)\nlog("VERDICT")\nlog("=" * 78)\ndef at(rows, n):\n    for r in rows:\n        if r.get("N") == n and "median_ms" in r: return r\n    return None\n\nbase = sweeps["baseline"]["points"]\nb4096 = at(base, 4096)\nlog("  the load-bearing number: framework DEVICE time at N=4096, E=2^24")\nif b4096: log(f"    SPLIT=256  {b4096[\'median_ms\']:8.3f} ms  {b4096[\'kernels\']} kernels")\nfor k in SPLIT_VALUES:\n    r = at(sweeps.get(f"split{k}", {}).get("points", []), 4096)\n    if r: log(f"    SPLIT={k:<5} {r[\'median_ms\']:8.3f} ms  {r[\'kernels\']} kernels")\n    elif f"split{k}" in sweeps: log(f"    SPLIT={k:<5} MISSING")\ncl = at(sweeps.get("control_last", {}).get("points", []), 4096)\nif cl:\n    log(f"    control    {cl[\'median_ms\']:8.3f} ms  (unmodified, re-measured LAST)")\n    if b4096:\n        d = abs(cl["median_ms"] - b4096["median_ms"]) / b4096["median_ms"]\n        log(f"    baseline vs control: {d*100:.2f}% apart"\n            f"{\'  -- SESSION DRIFT, F1 VOID\' if d > 0.05 else \'  -- within 5%, control holds\'}")\n\nlog("")\nlog("  F1  does the time at N=4096 TRACK SPLIT?")\ngot = {}\nif b4096: got[INSTALLED] = b4096["median_ms"]\nfor k in SPLIT_VALUES:\n    r = at(sweeps.get(f"split{k}", {}).get("points", []), 4096)\n    if r: got[k] = r["median_ms"]\ndrift_ok = (cl is None or b4096 is None\n            or abs(cl["median_ms"] - b4096["median_ms"]) / b4096["median_ms"] <= 0.05)\nif len(got) == 3 and drift_ok:\n    order = [got[64], got[256], got[1024]]\n    monotonic = order[0] < order[1] < order[2]\n    log(f"      SPLIT=64 {order[0]:.3f} < SPLIT=256 {order[1]:.3f} < SPLIT=1024 {order[2]:.3f}"\n        f"  -> {monotonic}")\n    log(f"      F1 {\'PASSES: the time tracks the literal\' if monotonic else \'FIRED: the literal does not move the time\'}")\n    if monotonic:\n        log(f"      ratio 1024/64 = {order[2]/order[0]:.2f}x   (the model predicts 16x on work;"\n            f" the realised ratio is what the card does with it)")\nelif not drift_ok:\n    log("      F1 VOID: the unmodified control moved more than 5% between the start")\n    log("         and the end of the session, so a difference attributed to SPLIT")\n    log("         could equally be a session difference. Reported as such.")\nelse:\n    log("      F1 INDETERMINATE: not all three SPLIT values completed at N=4096")\nlog("")\nlog("  F2  is the peak at 2*SPLIT?")\nfor key, sw in sweeps.items():\n    pts = [p for p in sw["points"] if "median_ms" in p]\n    if not pts: continue\n    sp = sw["split"]\n    below = [p for p in pts if p["N"] <= 2 * sp and "median_ms" in p]\n    above = [p for p in pts if p["N"] > 2 * sp and "median_ms" in p]\n    pk = max(pts, key=lambda p: p["median_ms"])\n    log(f"    SPLIT={sp:<5} slowest N={pk[\'N\']} ({pk[\'median_ms\']:.3f} ms);"\n        f" 2*SPLIT={2*sp}; max below={max((p[\'N\'] for p in below), default=None)};"\n        f" min above={min((p[\'N\'] for p in above), default=None)}")\nlog("")\nlog("  F5  does the cost RISE into the switch and DROP after it? (SPLIT=256)")\nif base:\n    b = {p["N"]: p["median_ms"] for p in base if "median_ms" in p}\n    for pair in ((256, 512), (384, 640), (512, 640), (512, 1024)):\n        if pair[0] in b and pair[1] in b:\n            log(f"    N={pair[0]:<5} {b[pair[0]]:8.3f} ms   ->  N={pair[1]:<5} {b[pair[1]]:8.3f} ms"\n                f"   ratio {b[pair[1]]/b[pair[0]]:.3f}")\n\nlog("")\nlog("=" * 78)\nlog(f"probe finished, {left():.0f}s of budget unused")\nlog("=" * 78)\n\nRESULT = {\n    "probe": "axis/splitcausal_t4probe.py",\n    "dev": DEV,\n    "tinygrad_file": tinygrad.__file__,\n    "op_py": str(OP_PY), "op_py_sha256": _h,\n    "installed_SPLIT": INSTALLED,\n    "predicted_first_two_stage_N": 2 * INSTALLED + 1,\n    "tensor_wraps_mixin": Tensor.__dict__.get("_split_cumalu") is not M.OpMixin.__dict__["_split_cumalu"],\n    "E": E, "dense_N": DENSE,\n    "one_vs_two": s1, "sweeps": sweeps,\n    "n4096_by_split": got,\n    "control_last_n4096_ms": cl["median_ms"] if cl else None,\n    "session_drift_pct": (None if (cl is None or not b4096)\n                          else round(100*abs(cl["median_ms"]-b4096["median_ms"])/b4096["median_ms"], 3)),\n    "budget_s": BUDGET_S, "wall_s": round(time.time() - T0, 1),\n    "patch_diff_s64": (sweeps.get("split64") or {}).get("patch_diff"),\n}\nopen(os.environ.get("SPLITCAUSAL_OUT", "splitcausal_t4.json"), "w").write(json.dumps(RESULT, indent=2, default=str))\nlog("wrote", os.environ.get("SPLITCAUSAL_OUT", "splitcausal_t4.json"))\n'
_SHA = '90c25ec097ce6df0808f9f1429af5ae1cb57a26603023df43d4419e33bc2cff1'
assert hashlib.sha256(_PAYLOAD.encode()).hexdigest() == _SHA, 'payload changed in transit'
exec(compile(_PAYLOAD, '<splitcausal-t4-probe>', 'exec'))
